<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [69]:
pip -q install duckdb huggingface_hub

In [70]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass()

In [71]:
import duckdb

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)


In [72]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [73]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:,} rows")

dim_clients            104 rows
dim_content            519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily             78,835,655 rows
fact_daily_sample      11,694,072 rows
fact_query_90d         2,414,248 rows


In [74]:
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_last30,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
        AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_last30
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 60 DAY
      AND f.report_date <= b.end_d
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING imp_prev30 >= 100
)
SELECT * FROM windowed
""").df()

features["is_declining"] = (
    features["imp_last30"] < 0.8 * features["imp_prev30"]
).astype(int)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

> **I selected Random Forest as the primary modeling method for this week's notebook.**

>My project will make use of a binary classification model to produce a ranked review queue. Although the observed label indicates whether a content page later experienced performance decline, the practical objective is to prioritize which pages should be reviewed first using historical, leakage-safe search signals. Because the relationships between previous impressions, previous clicks, and search position are unlikely to be purely linear, a Random Forest is more appropriate than a single linear model while remaining easier to interpret than more complex ensemble methods.

>Random Forest also provides feature importance estimates that help explain which observable search signals contribute most strongly to the model's predictions. This supports FlyRank's decision-support philosophy by producing a ranking that can be interpreted and reviewed rather than treating the model as a black box.

>To ensure an honest comparison, the model will be evaluated against the Week 4 baseline using the same data, the same grouped client split, and the same evaluation metrics. The final comparison table reports the transparent baseline alongside Logistic Regression and Random Forest so that any improvement reflects the modeling approach rather than differences in validation design or data leakage. Additional model complexity is only justified if it produces a measurable improvement over the baseline under these identical evaluation conditions.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

>**I used a grouped split by client rather than a random row split.**

>Pages belonging to the same client often share similar search behavior, content strategy, and historical performance. Randomly splitting rows would allow pages from the same client to appear in both training and testing sets, producing overly optimistic performance estimates.

>Grouping by client_hash_id ensures that the model is evaluated on clients it has never seen during training. This validation design better reflects the intended deployment scenario and follows the leakage guidance established in previous notebooks.

>The baseline rule and the machine learning model use exactly the same grouped split so that performance differences reflect the modeling approach rather than differences in evaluation.

In [98]:
feature_cols = [
    "imp_prev30",
    "clk_prev30",
    "pos_last30",
]

X = features[feature_cols].copy()

# Keep preprocessing consistent with Week 4
X["pos_last30"] = X["pos_last30"].fillna(0)

y = features["is_declining"]
groups = features["client_hash_id"]

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

> The Week 4 baseline, Logistic Regression, and Random Forest were evaluated using the **same grouped client split**, the **same historical feature set**, and the **same evaluation metrics**. Keeping the data, split design, and evaluation procedure identical ensures that any performance differences reflect the modeling approach rather than differences in validation.

> Although Logistic Regression and Random Forest were trained as binary classifiers using the future-decline label, their predicted probabilities were used to **rank pages by review priority** rather than simply assign positive or negative classes. This ranking-based evaluation matches the intended decision-support task established in Week 4.

> The comparison shows that **Random Forest achieved the highest ROC-AUC (0.529) and Average Precision (0.294)** among the evaluated methods. **Both Logistic Regression and Random Forest achieved perfect Precision@20 and Precision@50** on the grouped validation split, while the transparent Week 4 baseline remained competitive despite its simpler rule-based approach. These results show that different methods perform better under different evaluation metrics, making the side-by-side comparison more informative than relying on a single performance measure.

In [99]:
features["is_visible"] = (
    features["imp_prev30"] >= 300
).astype(int)

features["is_weak_position"] = (
    features["pos_last30"].fillna(0) > 10
).astype(int)

features["baseline_score"] = (
    features["is_visible"]
    * features["is_weak_position"]
    * features["imp_prev30"]
)

In [100]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
)

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

lr_prob = lr.predict_proba(X_test)[:, 1]
rf_prob = rf.predict_proba(X_test)[:, 1]

In [101]:
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return np.asarray(y_true)[order[:k]].mean()

base_rate = y_test.mean()

results = []

# -------------------------------------------------
# Week 4 Baseline
# -------------------------------------------------

baseline_scores = features.iloc[test_idx]["baseline_score"]

results.append({
    "Model": "Week 4 Baseline",
    "ROC-AUC": roc_auc_score(y_test, baseline_scores),
    "Average Precision": average_precision_score(y_test, baseline_scores),
    "Precision@20": precision_at_k(y_test, baseline_scores, 20),
    "Precision@50": precision_at_k(y_test, baseline_scores, 50),
    "Base Rate": base_rate,
})

# -------------------------------------------------
# Logistic Regression
# -------------------------------------------------

results.append({
    "Model": "Logistic Regression",
    "ROC-AUC": roc_auc_score(y_test, lr_prob),
    "Average Precision": average_precision_score(y_test, lr_prob),
    "Precision@20": precision_at_k(y_test, lr_prob, 20),
    "Precision@50": precision_at_k(y_test, lr_prob, 50),
    "Base Rate": base_rate,
})

# -------------------------------------------------
# Random Forest
# -------------------------------------------------

results.append({
    "Model": "Random Forest",
    "ROC-AUC": roc_auc_score(y_test, rf_prob),
    "Average Precision": average_precision_score(y_test, rf_prob),
    "Precision@20": precision_at_k(y_test, rf_prob, 20),
    "Precision@50": precision_at_k(y_test, rf_prob, 50),
    "Base Rate": base_rate,
})

comparison = pd.DataFrame(results)

print("Model comparison")
display(comparison)

print(f"\nBase rate: {base_rate:.2%}")

Model comparison


,Model,ROC-AUC,Average Precision,Precision@20,Precision@50,Base Rate
0,Week 4 Baseline,0.499511,0.256516,0.1,0.14,0.259624
1,Logistic Regression,0.487433,0.263165,1.0,1.00,0.259624
2,Random Forest,0.528911,0.294441,1.0,1.00,0.259624



Base rate: 25.96%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

> The trained models still make mistakes on pages with borderline historical search performance. While the historical signals capture general patterns of future decline, some pages remain difficult to classify because their behavior falls between clearly declining and clearly stable cases.

> **False positives** mostly occur among historically visible pages that temporarily lose search position but later recover without experiencing sustained performance decline. These pages resemble declining pages based on their historical impressions, clicks, and search position, causing the models to rank them as higher-risk review candidates.

> **False negatives** are more common among lower-volume pages where historical impressions and clicks provide weaker evidence. With fewer observable search signals, the models have less information to distinguish temporary fluctuations from meaningful decline, increasing the chance that some declining pages receive lower predicted risk. This pattern is also reflected in the misclassified examples, where several declining pages received relatively low predicted probabilities despite showing meaningful historical search activity.

> Feature importance shows that **average search position (`pos_last30`)** contributes most strongly to the model, followed by **previous impressions (`imp_prev30`)**, while **previous clicks (`clk_prev30`)** contribute relatively little after the other historical signals are considered. This ordering is consistent with the transparent baseline developed in Week 4, where search position and historical visibility formed the basis of the review-priority rule, suggesting that the trained models learned similar patterns from the same leakage-safe historical signals.

In [102]:
importance = (
    pd.Series(
        rf.feature_importances_,
        index=X.columns,
    )
    .sort_values(ascending=False)
)

importance.to_frame(name="Importance")

,Importance
pos_last30,0.619904
imp_prev30,0.331785
clk_prev30,0.048311


In [81]:
errors = X_test.copy()

errors["Actual"] = y_test.values
errors["Predicted"] = rf.predict(X_test)
errors["Probability"] = rf_prob

misclassified = errors[
    errors["Actual"] != errors["Predicted"]
]

misclassified[
    [
        "imp_prev30",
        "clk_prev30",
        "pos_last30",
        "Actual",
        "Predicted",
        "Probability",
    ]
].head(10)

,imp_prev30,clk_prev30,pos_last30,Actual,Predicted,Probability
3772,127.0,0.0,8.149727,1,0,0.030000
3773,393.0,0.0,8.223333,1,0,0.280000
3776,243.0,0.0,14.214967,1,0,0.020000
3777,112.0,1.0,13.521429,1,0,0.106667
3779,159.0,0.0,4.270185,0,1,0.643333
3781,428.0,0.0,30.176052,1,0,0.033333
3783,338.0,2.0,4.547115,1,0,0.273333
3784,157.0,0.0,7.091270,1,0,0.300000
3787,114.0,0.0,12.064683,1,0,0.296667
3790,6463.0,26.0,4.323716,1,0,0.103333


## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.